In [ ]:
from google.colab import files
import os

# Install the Kaggle API token.
print("Please select your kaggle.json file.:")
uploaded = files.upload()

# Set the necessary folder permissions.
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("The Kaggle API has been successfully configured!")

In [ ]:
# Download the dataset from Kaggle at high speed.
!kaggle datasets download -d abdallahalidev/plantvillage-dataset

# Extract to folder
print("Data is being extracted from the zip file, please wait...")
!unzip -q plantvillage-dataset.zip -d ./plant_data
print("The dataset has been successfully prepared!")

In [ ]:
import os
import shutil
import random

# Our source folder is now directly the 'color' folder.
source_dir = '/content/plant_data/plantvillage dataset/color'
target_dir = '/content/plant_structured_data'
train_dir = os.path.join(target_dir, 'train')
val_dir = os.path.join(target_dir, 'val')

# Let's start with a clean slate.
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# List classes
classes = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]

print(f"🔄 The total {len(classes)} plant classes are divided as 80% Train / 20% Val...")

for cls in classes:
    cls_source = os.path.join(source_dir, cls)

    # Create target folders
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)

    # Take and mix pictures
    images = [f for f in os.listdir(cls_source) if os.path.isfile(os.path.join(cls_source, f))]
    random.shuffle(images)

    split_idx = int(len(images) * 0.8)
    train_images = images[:split_idx]
    val_images = images[split_idx:]

    # Copy
    for img in train_images:
        shutil.copy(os.path.join(cls_source, img), os.path.join(train_dir, cls, img))
    for img in val_images:
        shutil.copy(os.path.join(cls_source, img), os.path.join(val_dir, cls, img))

print("✅ The dataset has been seamlessly converted to YOLOv8 format!")

In [ ]:
%pip install ultralytics

import os
from ultralytics import YOLO

# 1. We are clarifying the folder structure and the full address.
base_data_path = '/content/plant_structured_data'

# Check if the folder structure was different after yesterday's unzip operation:
# The `os.listdir(base_data_path)` command should show folders like `train/val` or classes directly.
# YOLOv8 Classification requires the main folder containing the classes directly.
print(f"Searching for a dataset: {base_data_path}")
print(f"Folder contents: {os.listdir(base_data_path)}")

# 2. We are loading the model with fresh weights.
model = YOLO('yolov8n-cls.pt')

# 3. We're triggering training!
print("🚀 Model training begins...")
model.train(
    data=base_data_path,  # We provided the full address as a Python variable.
    epochs=10,            # 20 epochs for a balance of speed and success.
    imgsz=224,            # Standard resolution for classification
    device=0              # For Colab to use the T4 GPU
)